# Retrieval ladder

Ask your coding assistant to run real retrieval experiments, then inspect what each method found. Compare dense search, BM25, fusion, reranking, and query expansion on the same questions.

## Learn | Create | Grow

### Learn
Inspect the evidence labels and compare the retrieval methods on identical chunks.

### Create
Measure hit rate, MRR, and search time. Use the per-case results to decide where extra retrieval work helps.

### Grow
Test the smallest retrieval pipeline that meets your quality and latency needs.

**Estimated time:** 40 minutes

Reads: corpus, vibe checks, and existing eval cases when available.

Writes: eval cases and measured ladder only when you ask the tool to save them.

## Setup

Open the repository in a coding assistant with file and terminal access. Follow the messages below; the README documents the experiment tools and environment setup.

Product documentation: [Codex](https://developers.openai.com/codex/), [Claude Code](https://code.claude.com/docs/en/overview), [VS Code Copilot](https://code.visualstudio.com/docs/agents/overview).

### Ask your assistant

> Read `06_Advanced_Retrieval/README.md` and inspect `06_Advanced_Retrieval/retrieval_tools.py`. Use the inspect command to show which corpus is active, how many pages it has, and which pages are excluded. Check the existing evidence cases too. Do not run a model experiment yet.

Every rung uses the same chunks. The tools exclude the wiki index and the vibe-check answer-key page from retrieval. Their results say whether inputs came from your workspace or the seed example.

Your assistant should execute the repo tool, not substitute its own file search for dense search or BM25. Ask it to show the tool result if the explanation is unclear.

### Terms you will use

| Term | Meaning |
|---|---|
| Dense search | Embedding similarity, useful for related meanings |
| BM25 | Word matching weighted by rarity, frequency, and document length |
| RRF | Combine ranks from several lists |
| Cross-encoder | Score a question and candidate passage together |
| Multi-query | Search several rewrites, then combine the candidates |
| Hit rate | Fraction of questions with a labelled page in the top results |
| MRR | Average reciprocal rank of the first labelled result |

# Learn

## Task 1 of 4 — Label the evidence

A score needs an answer key. For each question, identify pages that contain evidence, not just matching words. Label pages rather than chunk IDs so the labels can survive a chunk-size change.

### Ask your assistant

> Use `06_Advanced_Retrieval/retrieval_tools.py` to propose evidence-page labels for the current vibe checks. Compare them with the existing eval cases. Show one question, its proposed pages, and the relevant source text. Wait for me to review the labels before treating new proposals as the answer key. Do not save anything yet.

Check the proposed labels against the full pages. The labeling tool sends its model only the first 240 characters of each page, so it may miss relevant evidence later in the text. A missing label can make a useful result score as a miss. An empty page list means the case is excluded from retrieval scoring, not that the system passed it.

An out-of-scope question may still have a relevant policy page explaining the refusal. Review that distinction rather than automatically deleting its labels.

## Task 2 of 4 — Dense and sparse

Dense search can find related meanings when words differ. BM25 can reward exact terms such as setting names and error strings. Test their actual rankings before deciding which works better.

### Ask your assistant

> Use `06_Advanced_Retrieval/retrieval_tools.py` to compare dense and BM25 for: My VPN connects but I cannot reach staging.
>
> Keep the default chunks and top four results. Show the page names and matching excerpts, including the scratch BM25 ranking. Explain what the code does differently from the library BM25. Use the returned results rather than guessing which retriever should win.

In the source, BM25 combines term rarity, frequency saturation, and document-length correction. The scratch version and library use different IDF formulas, so their rankings may differ even with the same tokenizer.

Dense and sparse scores are on different scales. Compare ranks and evidence quality; do not add the raw scores together.

<details>
<summary>Recorded experiment: one question</summary>

Seed corpus, 28 pages, 65 chunks, top four. Recorded 2026-09-15 (UTC).

| Method | Top pages, in rank order |
|---|---|
| dense | prompts/meta-prompt-applied.md; transcripts/t01.md; kb/vpn.md; transcripts/t01.md |
| bm25 | transcripts/t01.md; transcripts/t02.md; transcripts/t05.md; charter.md |

These are actual tool results. Repeated page names are different chunks from the same page. Page and chunk references, text hashes, and settings are in [the experiment record](data/recorded_experiments.json).

</details>

### ❓ Question
Pick the question where dense and BM25 disagree most. Which words in the question did BM25 lock onto, and which meaning did dense search chase instead?

Answer:

## Task 3 of 4 — Fuse, rerank, expand

RRF combines ranks using one divided by sixty plus the rank. Reranking reorders a shortlist; it cannot recover a page absent from that shortlist. Multi-query asks the question several ways before combining and reranking candidates.

### Ask your assistant

> Use `06_Advanced_Retrieval/retrieval_tools.py` to run all five rungs with default settings for: My VPN connects but I cannot reach staging. Show the dense and BM25 candidate lists used by RRF, the shortlist before reranking, and the generated query rewrites. Compare the final top four results. Identify a page that moved up or disappeared and trace what happened using the tool output.

Keep the question, chunks, and candidate count fixed while comparing rungs. Check whether a rewrite preserves the original intent and whether the evidence made it into the candidate pool.

More processing does not guarantee a better ranking. Its benefit depends on the corpus, the question, and the model.

# Create

## Task 4 of 4 — Score the ladder

Hit rate asks whether a labelled page appears among the top results. Reciprocal rank rewards finding it sooner: rank one gives one, rank four gives one quarter, and a miss gives zero. Average those values to obtain MRR.

### Ask your assistant

> Use `06_Advanced_Retrieval/retrieval_tools.py` to score all five rungs on the active evidence cases and top-four setting. Use my reviewed labels if I provided them; otherwise use the existing cases and state their source. Show hit rate, MRR, search time, and the per-case reciprocal-rank matrix. List any skipped cases. Do not change labels to improve a score or save workspace artifacts.

<details>
<summary>Recorded experiment: measured ladder</summary>

5 seed cases; top four; embeddings text-embedding-3-small; rewrites gpt-4.1-mini. Recorded 2026-09-15 (UTC).

| Retriever | Hit rate | MRR | Search ms/query |
|---|---:|---:|---:|
| dense | 0.800 | 0.533 | 192.06 |
| bm25 | 0.200 | 0.067 | 0.26 |
| hybrid_rrf | 0.400 | 0.150 | 201.25 |
| cross_encoder | 0.200 | 0.200 | 310.67 |
| multi_query | 0.200 | 0.200 | 1798.72 |

Search time excludes model loading and index construction. This is one sequential run, not a latency benchmark. The full record contains labels, ranked chunks, rewrites, timings, and model settings.

</details>

Read the per-case results before the averages. Page-based labels can miss duplicated evidence on another page; inspect excerpts before concluding a retriever is useless.

Measure latency repeatedly before choosing a configuration. Query embeddings and query expansion are included in search time; initial model loading and index construction are reported separately.

### ❓ Question
Read the per-case matrix, not the averages. Which case separates the rungs, and would you pay the top rung's latency for that one case?

Answer:

### Try your own question

Tell your assistant your question and the pages you identified as evidence. Ask it to use the same tool and settings for each retriever. The source accepts reviewed cases and different chunk sizes or candidate counts, so you can change one variable and rerun without rewriting the algorithms.

A finding applies to the cases you tested. Failing to find a separating question does not establish that dense retrieval is sufficient for every future question.

## Your turn

Write one question over your corpus where exactly one retriever gets the right page at rank 1. Label its pages, run it through every rung, and write one sentence on why that rung won. If you cannot find such a question, that is a finding too: dense search is enough for this corpus.

# Grow

## From prototype to production

| Experiment | Production requirement |
|---|---|
| Proposed page labels | Reviewed evidence and new cases from real user questions |
| An index rebuilt for each run | Incremental indexing and a refresh policy |
| One chunk size and candidate count | Settings evaluated on held-out cases |
| A local reranker | Model versions and a serving latency budget |
| One pipeline for every question | Routing based on measured question types |
| A single score table | Regression checks over time |

## Responsible controls

- Keep the answer key out of the retrieval index and review labels before scoring.
- Record the corpus, models, settings, and actual results; never invent a ranking or metric.
- Compare quality and repeated latency measurements before adopting a more expensive rung.

## Grow further

- Change chunk size while keeping page labels fixed, then measure again.
- Compare exact-token questions with paraphrases before trying a router.
- Cache embeddings or rewrites and measure the effect on latency.

<details>
<summary>About the recorded experiments</summary>

The recordings were produced by `retrieval_tools.py` on the seed corpus. Proposed labels are included as proposals, not treated as reviewed truth. Scoring used the existing seed eval cases. No live workspace artifacts were changed for these recordings.

If you want to keep an experiment for later workflows, ask your assistant to run the score tool with its save option. That writes only the actual cases and measured ladder through the workspace helper.

</details>